In [17]:
import os
from src.db_client import ClientCreator

host = "10.245.12.58"
host_slurm = "arctrdcn018.rs.gsu.edu"
db_host = host_slurm if os.environ.get("SLURM_JOB_ID") else host
crop_tensor = False
client_creator = ClientCreator(
    db_host, crop_tensor=crop_tensor
)

db_name = databases = "multimodalSubnetworks"
db_collection = collections = "fbirn_falff"
db_fields = ["image", "gender"]
index_id = 'id'
numcubes = 1
cubesizes = 256
subvolume_shape = [cubesizes] * 3
shape = subvolume_shape[0]
numvolumes = 4
prefetches = 16

client_creator.set_database(databases)
client_creator.set_collection(collections)
client_creator.set_num_subcubes(numcubes)
client_creator.set_shape(subvolume_shape)

funcs = {
    "createclient": client_creator.create_client,
    "createVclient": client_creator.create_client,
    "mycollate": client_creator.mycollate,
    "mycollate_full": client_creator.mycollate_full,
    "mytransform": client_creator.mytransform,
}

from mindfultensors.mongoloader import MongoDataset, MongoClient
from mindfultensors.utils import unit_interval_normalize, DBBatchSampler
from catalyst.data import BatchPrefetchLoaderWrapper
from torch.utils.data import DataLoader

client = MongoClient("mongodb://" + db_host + ":27017")
db = client[db_name]
posts = db[db_collection + ".bin"]


/data/users2/ppopov1/miniconda/envs/catalyst12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/data/users2/ppopov1/miniconda/envs/catalyst12/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. Th

In [ ]:
train_dataset = MongoDataset(
    [0, 1, 2, 3, 4, 5, 6, 7], 
    funcs["mytransform"],
    None,
    db_fields,
    normalize=unit_interval_normalize,
    id=index_id,
)

train_sampler = DBBatchSampler(train_dataset, batch_size=numvolumes)

collate = (
    funcs["mycollate_full"]
    if shape == 256
    else funcs["mycollate"]
)

train_dataloader = BatchPrefetchLoaderWrapper(
            DataLoader(
                train_dataset,
                sampler=train_sampler,
                collate_fn=collate, # COLLATOR TRANSFORMS DATA USING mindfultensors.mongoloader.mcollate
                pin_memory=True,
                worker_init_fn=funcs["createclient"],
                persistent_workers=True,
                prefetch_factor=2,
                num_workers=4,  # self.prefetches,
                # prefetch_factor=None,
                # num_workers=1,  # self.prefetches,
            ),
            num_prefetches=prefetches,
        )

In [25]:
shape == 256

True